<a href="https://colab.research.google.com/github/onuralpArsln/MlAiTutorialProjects/blob/main/13-RAG/advancedRag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bölüm 1: Sentetik Veri Oluşturma (Bilim Kurgu Arşivi)

Bu hücre, RAG sistemimizin kaynağı olacak 'Xylos-9 Enerji Raporu 2245' dökümanını içerir. Bu bilgiler tamamen uydurmadır.

In [2]:
knowledge_base = """
# DOSYA NO: 2245-ALPHA-X
# KONU: NİTRAT SİLİKA JEL (NSJ) VS. MİKRO-DİFÜZYON REAKTÖRLERİ (MDR)

## 1. Teknolojik Karşılaştırma
Nitrat Silika Jel (NSJ) hücreleri, %98.4 oranında 'Zeta-Foton' verimliliği ile çalışırken, Mikro-Difüzyon Reaktörleri (MDR) kuantum sızıntısı nedeniyle %62'de kalmaktadır. Ancak MDR'lerin 'hiper-iyon' kararlılığı 400 yıldır bozulmamıştır.

## 2. Pazar Hakimiyeti ve Şirketler
Bu teknolojilerin üretiminde 3 ana maden devi ve 2 teknoloji markası çatışmaktadır:

### Maden Devleri:
1. **Aether-Corp:** 'Gelişmiş Kagara Adaları'ndaki lityum-nitrat yataklarının %80'ini kontrol ediyor.
2. **Nebula Mining:** MDR yakıtı olan 'Void-Kristali'nin tek tedarikçisi.
3. **Iron-Titan:** Yer altı şehirlerinde robotik iş gücüyle üretim yapan, etik dışı yöntemleriyle bilinen dev.

### Öncü Markalar:
- **Xenon Dynamics:** NSJ teknolojisinin mucidi. CEO'su Valerius Kray, MDR teknolojisini 'antika bir fener' olarak nitelendiriyor.
- **Nova-Core:** MDR sistemlerini savunan, gelenekçi ama askeri lobisi çok güçlü olan marka.

## 3. Siyasi Oyunlar ve Sömürü
'Gelişmiş Kagara Adaları' ve 'Karanlık Kıta Orizon' gibi geri kalmış bölgeler, bu enerji savaşı yüzünden ekolojik yıkıma uğradı. Aether-Corp, Orizon yerlilerine enerji vaadi verip karşılığında atmosferik oksijenlerini depolayarak satmaktadır. Xenon Dynamics ise Nova-Core'un üretim hatlarını sabote etmek için 'Nano-Virüs' protokollerini sızdırmıştır.

## 4. Kritik İstatistikler
- Bir NSJ hücresi, 4.2 gigavat-saat enerjiyi bir toplu iğne ucu kadar alanda depolar.
- MDR patlamaları, çevredeki zaman akışını %3 oranında yavaşlatır (Zaman Kayması Etkisi).
"""

with open('bilim_kurgu_arsivi.txt', 'w', encoding='utf-8') as f:
    f.write(knowledge_base)

print("Uydurma döküman başarıyla oluşturuldu: bilim_kurgu_arsivi.txt")

Uydurma döküman başarıyla oluşturuldu: bilim_kurgu_arsivi.txt


## Bölüm 2: TensorFlow ve LangChain ile Gelişmiş RAG Kurulumu

Bu bölümde aşağıdaki bileşenleri kuracağız:
1. **TensorFlow Hub Embeddings**: Metinleri vektörleştirmek için.
2. **ChromaDB**: Vektör veritabanı.
3. **BM25**: Kelime bazlı arama (Hybrid yapı için).
4. **Cross-Encoder**: Re-ranking işlemi için.

In [1]:

!pip install -q -U transformers sentence-transformers rank_bm25 tensorflow-hub tensorflow-text


In [3]:
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text as text
import numpy as np
from rank_bm25 import BM25Okapi

# 1. Dokümanı Hazırla
with open('bilim_kurgu_arsivi.txt', 'r', encoding='utf-8') as f:
    content = f.read()

# Basit chunking (Parçalara ayırma)
chunks = [c.strip() for c in content.split('\n\n') if len(c.strip()) > 10]

# 2. TensorFlow Hub Embedding Modeli (Universal Sentence Encoder)
print("Model yükleniyor...")
model_url = "https://tfhub.dev/google/universal-sentence-encoder-multilingual/3"
embed_model = hub.load(model_url)

# Vektörleri oluştur (TensorFlow üzerinden)
chunk_embeddings = embed_model(chunks)

# 3. BM25 Hazırlığı (Kelime bazlı arama için)
tokenized_corpus = [doc.lower().split() for doc in chunks]
bm25 = BM25Okapi(tokenized_corpus)

def get_tf_recommendations(query, k=5):
    # Vektör araması
    query_embedding = embed_model([query])
    # Cosine similarity hesapla
    similarities = tf.matmul(query_embedding, chunk_embeddings, transpose_b=True)
    v_scores = similarities.numpy()[0]

    # BM25 skorları
    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)

    # Hibrit Skor (Normalleştirilmiş)
    combined_scores = (v_scores * 0.7) + (bm25_scores / (np.max(bm25_scores) + 1e-6) * 0.3)
    top_indices = np.argsort(combined_scores)[::-1][:k]

    return [chunks[i] for i in top_indices]

print("TensorFlow tabanlı Hibrit sistem hazır.")

Model yükleniyor...
TensorFlow tabanlı Hibrit sistem hazır.


### Re-ranking (Yeniden Sıralama)

Arama sonuçlarını daha hassas hale getirmek için Cross-Encoder kullanıyoruz.

In [ ]:
import numpy as np
try:
    from sentence_transformers import CrossEncoder
    # Re-ranking için hafif bir model seçelim
    reranker_model = CrossEncoder('BAAI/bge-reranker-base')

    def search_and_rerank(query, top_k=3):
        # Önce hibrit arama yap
        initial_docs = get_tf_recommendations(query, k=10)

        # Çapraz skorlama (Re-ranking)
        pairs = [[query, doc] for doc in initial_docs]
        scores = reranker_model.predict(pairs)

        # Yeniden sırala
        reranked_indices = np.argsort(scores)[::-1][:top_k]
        return [initial_docs[i] for i in reranked_indices]

    print("Re-ranking fonksiyonu hazır.")
except Exception as e:
    print(f"Kurulum hatası: {e}")

In [ ]:
test_query = "Zaman akışındaki yavaşlama oranı nedir ve hangi teknoloji buna sebep olur?"

print(f"SORGU: {test_query}\n")

results = search_and_rerank(test_query)

for i, doc in enumerate(results):
    print(f"{i+1}. Sonuç:\n{doc}\n{'-'*20}")